# Proyecto Módulo 7 - Segmentador Inteligente de Clientes Minoristas

## Introducción

Retail Insights S.A., consultora especializada en análisis para cadenas minoristas y e-commerce, requiere desarrollar un sistema de segmentación de clientes que permita identificar grupos de comportamiento relevantes a partir de datos no etiquetados. La necesidad surge porque actualmente solo se utilizan variables básicas, como edad o compras totales, lo que limita la identificación de patrones ocultos y la personalización de campañas de marketing.

En este contexto, el presente proyecto tiene como propósito aplicar técnicas de aprendizaje no supervisado para construir una segmentación de clientes útil, interpretable y escalable. Para ello, se trabajará con algoritmos de clusterización, técnicas de reducción dimensional y métricas de evaluación que permitan comparar resultados y proponer usos comerciales concretos.

## Objetivo del proyecto

Desarrollar un sistema de segmentación de clientes a partir de un dataset no etiquetado utilizando técnicas de aprendizaje no supervisado, con el fin de identificar patrones ocultos de comportamiento, visualizar los grupos detectados y generar insights aplicables a campañas de retención, fidelización y ventas cruzadas.

## Metodología general

El desarrollo del proyecto seguirá cinco etapas principales:

1. Fundamentos del aprendizaje no supervisado  
2. Técnicas de clusterización  
3. Reducción dimensional y preprocesamiento  
4. Aplicación práctica de clusterización con Python  
5. Evaluación final e interpretación de resultados  

## Lección 1 - Fundamentos del Aprendizaje No Supervisado

### Objetivo de la lección

Comprender los fundamentos teóricos del aprendizaje no supervisado y sus diferencias con el aprendizaje supervisado, estableciendo una base conceptual adecuada para el desarrollo posterior del proyecto de segmentación.

### ¿Qué es el aprendizaje no supervisado?

El aprendizaje no supervisado corresponde a un conjunto de técnicas de machine learning que trabajan con datos no etiquetados. A diferencia del aprendizaje supervisado, en el cual existe una variable objetivo conocida, en este enfoque el propósito es descubrir estructuras ocultas, relaciones internas o agrupamientos naturales en los datos sin contar previamente con una clase o etiqueta de referencia.

### Principales tareas que resuelve

Las principales tareas del aprendizaje no supervisado son:

- **Clusterización:** agrupar observaciones similares entre sí, como en segmentación de clientes.
- **Reducción dimensional:** proyectar datos de alta dimensión en espacios más pequeños para facilitar análisis y visualización.
- **Reglas de asociación:** detectar combinaciones frecuentes o relaciones entre variables, por ejemplo en análisis de canastas de compra.

### Clasificación de técnicas por tipo

#### 1. Técnicas de clusterización
Buscan formar grupos homogéneos de observaciones a partir de su similitud.
Ejemplos:
- K-Means
- DBSCAN
- Agrupamiento jerárquico

#### 2. Técnicas de reducción dimensional
Permiten simplificar la estructura de los datos y hacerlos más interpretables.
Ejemplos:
- PCA
- t-SNE

#### 3. Técnicas de asociación
Buscan reglas o patrones frecuentes dentro del conjunto de datos.
Ejemplo:
- Apriori

### Diferencia con el aprendizaje supervisado

La diferencia principal entre ambos enfoques radica en la presencia o ausencia de etiquetas. En aprendizaje supervisado el modelo aprende a predecir una variable objetivo conocida, mientras que en aprendizaje no supervisado el objetivo es descubrir patrones, estructuras o segmentos sin una respuesta previamente definida.

### Casos reales de aplicación

El aprendizaje no supervisado se aplica en múltiples escenarios reales, entre ellos:

- Segmentación de clientes en retail y e-commerce
- Detección de comunidades en redes sociales
- Agrupamiento de documentos o noticias por similitud temática
- Detección de anomalías o comportamientos atípicos
- Reducción de dimensionalidad para visualización de datos complejos

### Relación con el proyecto actual

En este proyecto, el aprendizaje no supervisado se utilizará para segmentar clientes minoristas a partir de variables de comportamiento y perfil. El propósito será descubrir grupos con patrones de compra o características similares, de manera que la empresa pueda utilizar dichos segmentos para campañas más específicas y decisiones comerciales mejor informadas.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# =========================================================
# CONFIGURACIÓN DE RUTAS
# =========================================================
RUTA_RAW = Path("../data/raw")
RUTA_INTERIM = Path("../data/interim")
RUTA_OUTPUTS = Path("../outputs")

RUTA_INTERIM.mkdir(parents=True, exist_ok=True)
RUTA_OUTPUTS.mkdir(parents=True, exist_ok=True)

# =========================================================
# BÚSQUEDA AUTOMÁTICA DEL DATASET CSV
# =========================================================
archivos_csv = sorted(RUTA_RAW.glob("*.csv"))

if not archivos_csv:
    raise FileNotFoundError(
        "No se encontró ningún archivo CSV en ../data/raw/. "
        "Copia ahí el dataset del proyecto y vuelve a ejecutar."
    )

ruta_dataset = archivos_csv[0]
print(f"Dataset detectado: {ruta_dataset.name}")

# =========================================================
# CARGA DEL DATASET
# =========================================================
df = pd.read_csv(ruta_dataset)

print("\n===== DIMENSIONES DEL DATASET =====")
print(df.shape)

print("\n===== PRIMERAS 5 FILAS =====")
display(df.head())

print("\n===== ÚLTIMAS 5 FILAS =====")
display(df.tail())

print("\n===== TIPOS DE DATOS =====")
display(df.dtypes.to_frame("tipo_dato"))

print("\n===== VALORES NULOS POR COLUMNA =====")
display(df.isna().sum().to_frame("nulos").sort_values(by="nulos", ascending=False))

print("\n===== ESTADÍSTICAS DESCRIPTIVAS =====")
display(df.describe(include="all").T)

# =========================================================
# SEPARACIÓN INICIAL DE VARIABLES
# =========================================================
variables_numericas = df.select_dtypes(include=[np.number]).columns.tolist()
variables_categoricas = df.select_dtypes(exclude=[np.number]).columns.tolist()

resumen_variables = pd.DataFrame({
    "tipo_variable": (["Numérica"] * len(variables_numericas)) + (["Categórica"] * len(variables_categoricas)),
    "nombre_variable": variables_numericas + variables_categoricas
})

print("\n===== CLASIFICACIÓN DE VARIABLES =====")
display(resumen_variables)

print("\n===== RESUMEN INICIAL =====")
print(f"Cantidad de filas: {df.shape[0]}")
print(f"Cantidad de columnas: {df.shape[1]}")
print(f"Variables numéricas: {len(variables_numericas)}")
print(f"Variables categóricas: {len(variables_categoricas)}")
print(f"Total de valores nulos en el dataset: {int(df.isna().sum().sum())}")

# =========================================================
# EXPORTAR RESUMEN DE NULOS Y TIPOS
# =========================================================
resumen_nulos = df.isna().sum().reset_index()
resumen_nulos.columns = ["columna", "nulos"]
resumen_nulos["tipo_dato"] = resumen_nulos["columna"].map(df.dtypes.astype(str).to_dict())

resumen_nulos.to_csv(RUTA_INTERIM / "resumen_inicial_dataset.csv", index=False)

print("\nArchivo guardado: ../data/interim/resumen_inicial_dataset.csv")

Dataset detectado: Test.csv

===== DIMENSIONES DEL DATASET =====
(2627, 11)

===== PRIMERAS 5 FILAS =====


,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
0,458989,Female,Yes,36,Yes,Engineer,0.0,Low,1.0,Cat_6,B
1,458994,Male,Yes,37,Yes,Healthcare,8.0,Average,4.0,Cat_6,A
2,458996,Female,Yes,69,No,NaN,0.0,Low,1.0,Cat_6,A
3,459000,Male,Yes,59,No,Executive,11.0,High,2.0,Cat_6,B
4,459001,Female,No,19,No,Marketing,NaN,Low,4.0,Cat_6,A



===== ÚLTIMAS 5 FILAS =====


,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
2622,467954,Male,No,29,No,Healthcare,9.0,Low,4.0,Cat_6,B
2623,467958,Female,No,35,Yes,Doctor,1.0,Low,1.0,Cat_6,A
2624,467960,Female,No,53,Yes,Entertainment,NaN,Low,2.0,Cat_6,C
2625,467961,Male,Yes,47,Yes,Executive,1.0,High,5.0,Cat_4,C
2626,467968,Female,No,43,Yes,Healthcare,9.0,Low,3.0,Cat_7,A



===== TIPOS DE DATOS =====


...


===== ESTADÍSTICAS DESCRIPTIVAS =====


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ID,2627.0,NaN,NaN,NaN,463433.918919,2618.245698,458989.0,461162.5,463379.0,465696.0,467968.0
Gender,2627,2,Male,1424,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ever_Married,2577,2,Yes,1520,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,2627.0,NaN,NaN,NaN,43.649791,16.967015,18.0,30.0,41.0,53.0,89.0
Graduated,2603,2,Yes,1602,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Profession,2589,9,Artist,802,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Work_Experience,2358.0,NaN,NaN,NaN,2.552587,3.341094,0.0,0.0,1.0,4.0,14.0
Spending_Score,2627,3,Low,1616,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Family_Size,2514.0,NaN,NaN,NaN,2.825378,1.551906,1.0,2.0,2.0,4.0,9.0
Var_1,2595,7,Cat_6,1672,NaN,NaN,NaN,NaN,NaN,NaN,NaN



===== CLASIFICACIÓN DE VARIABLES =====


,tipo_variable,nombre_variable
0,Numérica,ID
1,Numérica,Age
2,Numérica,Work_Experience
3,Numérica,Family_Size
4,Categórica,Gender
5,Categórica,Ever_Married
6,Categórica,Graduated
7,Categórica,Profession
8,Categórica,Spending_Score
9,Categórica,Var_1



===== RESUMEN INICIAL =====
Cantidad de filas: 2627
Cantidad de columnas: 11
Variables numéricas: 4
Variables categóricas: 7
Total de valores nulos en el dataset: 526

Archivo guardado: ../data/interim/resumen_inicial_dataset.csv


### Hallazgos de la Lección 1

En esta primera etapa se estableció la base conceptual del proyecto, identificando las principales tareas del aprendizaje no supervisado y su pertinencia para un problema de segmentación de clientes. Además, se realizó una carga e inspección inicial del dataset, revisando dimensiones, tipos de variables, valores faltantes y estadísticas descriptivas preliminares. Esta revisión inicial permite comprender la estructura general de los datos y preparar el terreno para las siguientes etapas de preprocesamiento, reducción dimensional y clusterización.

## Lección 2 - Técnicas de Clusterización

### Objetivo de la lección

Comprender en profundidad las principales técnicas de clusterización y analizar su pertinencia para el problema de segmentación de clientes del proyecto.

### ¿Qué es la clusterización?

La clusterización es una técnica de aprendizaje no supervisado cuyo objetivo es agrupar observaciones similares entre sí y diferentes respecto de otros grupos. En el contexto comercial, esta técnica permite identificar segmentos de clientes con patrones de comportamiento comparables, aun cuando no exista una etiqueta previa que los clasifique.

### Escenarios donde se aplica clusterización

La clusterización se aplica en múltiples escenarios reales, entre ellos:

- segmentación de clientes en retail y e-commerce;
- agrupamiento de usuarios según hábitos de consumo;
- detección de comunidades en redes sociales;
- agrupamiento de documentos o textos por similitud temática;
- identificación de patrones anómalos o grupos atípicos;
- análisis de comportamiento en marketing digital y fidelización.

### Ventajas de la clusterización

Entre las principales ventajas de la clusterización se encuentran:

- permite descubrir patrones ocultos en datos no etiquetados;
- facilita la segmentación de clientes para campañas más específicas;
- ayuda a resumir grandes volúmenes de datos complejos;
- permite construir perfiles diferenciados útiles para retención, fidelización y ventas cruzadas;
- puede complementar procesos posteriores de modelado supervisado.

### Desventajas de la clusterización

Entre sus principales limitaciones se encuentran:

- los resultados dependen fuertemente del preprocesamiento y escalamiento de variables;
- distintos algoritmos pueden producir segmentaciones diferentes sobre el mismo dataset;
- algunos métodos requieren definir parámetros sensibles, como número de clústeres o radio de vecindad;
- los clústeres encontrados no siempre son directamente interpretables desde el negocio;
- la presencia de ruido, nulos y outliers puede afectar considerablemente la calidad del resultado.

### Comparación teórica entre algoritmos

#### 1. K-Means
K-Means busca particionar los datos en un número predefinido de clústeres minimizando la distancia entre cada punto y el centroide de su grupo. Es eficiente, rápido y muy utilizado en segmentación de clientes. Sin embargo, requiere definir previamente el número de clústeres y es sensible a outliers y a formas no esféricas en los datos.

#### 2. DBSCAN
DBSCAN agrupa observaciones según densidad, permitiendo detectar regiones densas y separar ruido o anomalías. Su principal fortaleza es que no requiere fijar previamente el número de clústeres y puede identificar outliers naturalmente. No obstante, puede presentar dificultades cuando las densidades de los grupos son muy diferentes o cuando los parámetros no son bien ajustados.

#### 3. Agrupamiento jerárquico
El clustering jerárquico construye una estructura de agrupamiento progresiva, representable mediante dendrogramas. Es útil para explorar relaciones entre observaciones y no exige necesariamente definir el número de clústeres desde el inicio. Sin embargo, su costo computacional puede ser mayor y puede volverse menos práctico en datasets muy grandes.

### Relación con el dataset del proyecto

En este dataset de clientes minoristas, la clusterización resulta pertinente porque existen variables numéricas y categóricas que describen edad, experiencia laboral, tamaño de familia, condición matrimonial, nivel de gasto y perfil ocupacional. Esto sugiere que podrían existir segmentos ocultos con características diferenciadas.

Para el desarrollo del proyecto:

- **K-Means** será útil como línea base para segmentación principal;
- **DBSCAN** permitirá evaluar la presencia de grupos densos y posibles anomalías;
- **agrupamiento jerárquico** será útil para explorar relaciones estructurales entre observaciones y apoyar la interpretación visual mediante dendrogramas.

### Decisión metodológica preliminar

Dado el tipo de problema planteado, la estrategia más adecuada será comparar los tres algoritmos sobre un dataset previamente limpio y transformado, utilizando reducción dimensional con PCA y t-SNE para facilitar la visualización de los grupos detectados.